# Phase 6 — XGBoost recovery model V1

This notebook audits the first predictive model for `P(recovery | customer history, payment context, intervention)`. Training is implemented in `ml.src.train_recovery_model`; policy simulation is implemented separately in `ml.src.evaluate_policy`.

The observed test metrics and the synthetic counterfactual policy simulation answer different questions and must not be conflated.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "config" / "dataset_manifest.yaml").exists():
            return candidate
    raise FileNotFoundError("RecoverAI repository root not found")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.src.model_pipeline import load_manifest, validate_dataset

DATASET_PATH = ROOT / "ml" / "data" / "processed" / "logging_policy_dataset.csv"
ARTIFACTS = ROOT / "ml" / "artifacts"
MANIFEST_PATH = ROOT / "ml" / "config" / "dataset_manifest.yaml"

data = pd.read_csv(DATASET_PATH, parse_dates=["prediction_time"])
manifest = load_manifest(MANIFEST_PATH)
metrics = json.loads((ARTIFACTS / "metrics_v1.json").read_text())
metadata = json.loads((ARTIFACTS / "model_metadata.json").read_text())
policy = json.loads((ARTIFACTS / "policy_evaluation_v1.json").read_text())
importance = json.loads((ARTIFACTS / "feature_importance_v1.json").read_text())

validate_dataset(data, manifest, source_path=DATASET_PATH)
print("Frozen dataset:", data.shape)
print("Raw model features:", metadata["raw_feature_count"])
print("One-hot transformed features:", metadata["transformed_feature_count"])

In [ ]:
split_audit = data.groupby("split").agg(
    rows=("payment_id", "size"),
    start=("prediction_time", "min"),
    end=("prediction_time", "max"),
    recovery_rate=("recovered", "mean"),
)
display(split_audit)

assert split_audit.loc["train", "end"] < split_audit.loc["validation", "start"]
assert split_audit.loc["validation", "end"] < split_audit.loc["test", "start"]
assert metadata["preprocessor_fit_split"] == "train"
assert metadata["model_fit_split"] == "train"
assert metadata["calibrator_fit_split"] == "validation"
assert metadata["test_used_for_fitting"] is False

In [ ]:
print("Numerical features:", len(manifest["numerical_features"]))
display(pd.Series(manifest["numerical_features"], name="numerical_feature").to_frame())
print("Categorical features:", len(manifest["categorical_features"]))
display(pd.Series(manifest["categorical_features"], name="categorical_feature").to_frame())

model_features = set(manifest["numerical_features"] + manifest["categorical_features"])
assert "recovered" not in model_features
assert "policy_probability" not in model_features
assert "fraud_flag" not in model_features
assert not any("counterfactual" in value.lower() for value in model_features)

## Reproduce training

Run from the repository root:

```powershell
.\ml\.venv\Scripts\python.exe -m ml.src.train_recovery_model
.\ml\.venv\Scripts\python.exe -m ml.src.evaluate_policy
```

The preprocessor and XGBoost model fit only the frozen training partition. Platt scaling fits only validation. The test partition is evaluated once after calibration is frozen.

In [ ]:
validation_metrics = pd.DataFrame({
    "before_calibration": metrics["validation_before_calibration"],
    "after_platt_fit": metrics["validation_after_platt_fit"],
})
validation_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for key, label in [
    ("validation_before", "Validation before calibration"),
    ("validation_after_platt_fit", "Validation after Platt fit"),
    ("test_after", "Test after calibration"),
]:
    bins = pd.DataFrame(metrics["calibration"][key])
    ax.plot(
        bins["mean_predicted_probability"],
        bins["observed_recovery_rate"],
        marker="o",
        label=label,
    )
ax.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")
ax.set(xlabel="Mean predicted recovery probability", ylabel="Observed recovery rate", title="Recovery probability calibration")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

In [ ]:
test_metrics = pd.DataFrame({
    "before_calibration": metrics["test_before_calibration"],
    "after_calibration": metrics["test_after_calibration"],
})
test_metrics

In [ ]:
intervention_metrics = pd.DataFrame(metrics["test_by_intervention"]).T
intervention_metrics[["rows", "positive_rate", "roc_auc", "pr_auc", "brier_score", "log_loss"]]

In [ ]:
top_importance = pd.DataFrame(importance).head(20).sort_values("importance")
ax = top_importance.plot.barh(
    x="feature",
    y="importance",
    figsize=(9, 7),
    legend=False,
    title="XGBoost V1 top feature importances",
)
ax.set(xlabel="Normalized XGBoost importance", ylabel="Transformed feature")
plt.tight_layout()
plt.show()

In [ ]:
policy_comparison = pd.DataFrame(policy["policies"]).T
policy_comparison[["recovery_rate", "recovered_payments", "recovered_amount"]].sort_values(
    "recovery_rate",
    ascending=False,
)

In [ ]:
print("RecoverAI action counts:", policy["recoverai_action_counts"])
print(
    "Uplift vs test always-retry:",
    policy["recoverai_uplift_vs_test_always_retry_percentage_points"],
    "percentage points",
)
print(
    "Recovered amount uplift:",
    policy["recoverai_recovered_amount_uplift_vs_test_always_retry"],
)
assert policy["recoverai_beats_test_always_retry"]

## Interpretation boundaries

- Observed test metrics evaluate predictions only for the interventions selected by the logging policy.
- RecoverAI policy value uses all four potential outcomes from the deterministic synthetic environment. It is a simulation result, not causal evidence from real interventions.
- The validation-after-calibration metrics reuse the calibration fitting set and are labeled as resubstitution; the frozen test metrics are the clean post-calibration estimate.
- `no_action` has only blocked fraud cases and no positive outcomes, so subgroup ROC-AUC and PR-AUC are undefined.
- Default XGBoost importance is descriptive, not a causal explanation. SHAP can be added later for case-level explanations.
- IPS and doubly robust estimators are intentionally deferred to Phase 7.